
# Построение RAG-системы по датасету Resume.csv с использованием LangChain, FAISS и Groq

**Тема проекта:** RAG (Retrieval-Augmented Generation) для поиска и анализа резюме  
**Стек:** langchain, langchain-community, langchain-huggingface,langchain-text-splitters, faiss-cpu, sentence-transformers, pandas, Groq


В этом проекте RAG-система строится **на датасете Resume.csv**, который содержит тексты резюме и профессиональные категории кандидатов.  
Система:

1. загружает и очищает резюме;
2. разбивает длинные тексты на чанки;
3. строит векторный индекс FAISS по эмбеддингам Sentence Transformers;
4. выполняет семантический поиск релевантных чанков;
5. передаёт найденный контекст в LLM через Groq;
6. тестируется на нескольких разнотипных вопросах с анализом качества.

## Почему выбран именно этот датасет

Resume.csv хорошо подходит для RAG-прототипа, потому что:
- в нём много документов (значительно больше минимально необходимых 100 записей);
- тексты длинные и полу-структурированные, поэтому чанкинг здесь действительно нужен;
- есть полезные метаданные (ID, Category);
- retrieval можно проверять на практических вопросах: навыки, опыт, категории, типичные компетенции, отсутствие информации.

Ниже используется поле **Resume_str**, потому что это уже извлечённый plain text, пригодный для чанкинга и семантического поиска.


In [2]:
!pip -q install -U datasets langchain langchain-community langchain-huggingface langchain-text-splitters langchain-groq faiss-cpu sentence-transformers

In [23]:
import os
import re
import pandas as pd
import numpy as np
import textwrap
from datasets import load_dataset
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_groq import ChatGroq

In [24]:
raw_df = pd.read_csv('Resume.csv')
print(raw_df.shape)
display(raw_df.head(3))

(2484, 4)


,ID,Resume_str,Resume_html,Category
0,16852973,HR ADMINISTRATOR/MARKETING ASSOCIATE\...,"<div class=""fontsize fontface vmargins hmargin...",HR
1,22323967,"HR SPECIALIST, US HR OPERATIONS ...","<div class=""fontsize fontface vmargins hmargin...",HR
2,33176873,HR DIRECTOR Summary Over 2...,"<div class=""fontsize fontface vmargins hmargin...",HR


In [25]:
required_cols = {"ID", "Resume_str", "Category"}
missing = required_cols - set(raw_df.columns)
df = raw_df[list(required_cols)].copy()
def clean_resume_text(text: str) -> str:
    text = "" if pd.isna(text) else str(text)
    text = re.sub(r"<[^>]+>", " ", text)# на случай HTML-артефактов
    text = re.sub(r"http\S+|www\.\S+", " ", text)# убираем URL
    text = text.replace("\xa0", " ")
    text = re.sub(r"\s+", " ", text).strip()
    return text
df["Resume_str"] = df["Resume_str"].astype(str).map(clean_resume_text)
df["Category"] = df["Category"].astype(str).str.strip()
df = df[df["Resume_str"].str.len() > 100].reset_index(drop=True)
print("После очистки:", df.shape)
print("Количество категорий:", df["Category"].nunique())
display(df[["ID", "Category", "Resume_str"]].head(3))

После очистки: (2483, 3)
Количество категорий: 24


,ID,Category,Resume_str
0,16852973,HR,HR ADMINISTRATOR/MARKETING ASSOCIATE HR ADMINI...
1,22323967,HR,"HR SPECIALIST, US HR OPERATIONS Summary Versat..."
2,33176873,HR,HR DIRECTOR Summary Over 20 years experience i...


In [26]:
SAMPLES_PER_CATEGORY = 12
balanced_df = (df.sort_values("ID").groupby("Category", group_keys=False).head(SAMPLES_PER_CATEGORY).reset_index(drop=True))
print("Размер подвыборки:", balanced_df.shape)
print("Записей >= 100:", balanced_df.shape[0] >= 100)
display(balanced_df["Category"].value_counts().sort_index())

Размер подвыборки: (288, 3)
Записей >= 100: True


,count
Category,
ACCOUNTANT,12
ADVOCATE,12
AGRICULTURE,12
APPAREL,12
ARTS,12
AUTOMOBILE,12
AVIATION,12
BANKING,12
BPO,12


Использование сбалансированной подвыборки по категориям позволяет уменьшить перекос в сторону наиболее частотных профессиональных групп. Благодаря этому retrieval оценивается на более репрезентативном наборе документов, а результаты тестирования становятся более интерпретируемыми.


### Обоснование подготовки данных

- Используется поле **Resume_str**, потому что оно уже содержит текст резюме в удобном виде.
- Поле **Category** сохраняется как метаданные: оно пригодится для анализа retrieval и объяснения ответов.
- Берётся **сбалансированная подвыборка** по категориям.


## 2. Подготовка документов и чанкинг.

Используем:
- chunk_size = 1200
- chunk_overlap = 200

Обоснование:
- резюме часто длинные и содержат список навыков + описание опыта;
- слишком маленький чанк разрушит связь между ролью, стеком и опытом;
- слишком большой чанк ухудшит retrieval;

In [27]:
documents = []
for row in balanced_df.itertuples(index=False):
    page_content = (f"Resume ID: {row.ID}\n"f"Candidate category: {row.Category}\n"f"Resume text:\n{row.Resume_str}")
    metadata = {"source": "Resume.csv","resume_id": int(row.ID),"category": row.Category,}
    documents.append(Document(page_content=page_content, metadata=metadata))
print("Количество исходных документов:", len(documents))
print(documents[0].metadata)
print(textwrap.shorten(documents[0].page_content, width=700, placeholder=" ..."))

Количество исходных документов: 288
{'source': 'Resume.csv', 'resume_id': 3547447, 'category': 'BANKING'}
Resume ID: 3547447 Candidate category: BANKING Resume text: MORTGAGE BANKING FORECLOSURE SPECIALIST Summary Ambitious, self-motivated professional with a passion for quality work. Seeking a baseline opportunity in Underwriting, Lending, Auditing, Quality Assurance, or Analyst roles. Possess large spectrum of experience in the financial industry. I am a fast learner who values my employer. Personal characteristics: detail-oriented, thorough, computer-savvy, loyal, persistent, adaptable, eager to learn. Accomplishments *Sharepoint, Early Resolution, FHA Connection, DOS LPS, MSP, CREDCO, RELS, Microsoft Word, Outlook, Live Meeting, Excel, Powerpoint, SLOAD, DAT and various other programs 3 ...


In [28]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1200,chunk_overlap=200,separators=["\n\n", "\n", ". ", "; ", ", ", " "])
chunked_documents = splitter.split_documents(documents)
for i, chunk in enumerate(chunked_documents):
    chunk.metadata["chunk_id"] = i
print("Количество чанков:", len(chunked_documents))
print("Среднее число чанков на документ:", round(len(chunked_documents) / len(documents), 2))

Количество чанков: 2265
Среднее число чанков на документ: 7.86


В результате разбиения было получено 2265 чанков, то есть в среднем около 7.86 чанка на одно резюме. Наверное, такое соотношение выглядит реалистичным для длинных резюме и подтверждает, что документы не остались целиком и при этом не были раздроблены чрезмерно мелко.

In [29]:
chunk_lengths = [len(doc.page_content) for doc in chunked_documents]
stats_df = pd.DataFrame({"metric": ["min", "25%", "50%", "mean", "75%", "max"],"value": [
        np.min(chunk_lengths),
        np.percentile(chunk_lengths, 25),
        np.percentile(chunk_lengths, 50),
        np.mean(chunk_lengths),
        np.percentile(chunk_lengths, 75),
        np.max(chunk_lengths),]})
display(stats_df)
print("Пример чанка:")
print("-" * 80)
print(textwrap.shorten(chunked_documents[0].page_content, width=1500, placeholder=" ..."))
print("-" * 80)
print("Метаданные чанка:", chunked_documents[0].metadata)

,metric,value
0,min,14.000000
1,25%,678.000000
2,50%,1069.000000
3,mean,868.654305
4,75%,1152.000000
5,max,1200.000000


Пример чанка:
--------------------------------------------------------------------------------
Resume ID: 3547447 Candidate category: BANKING Resume text:
--------------------------------------------------------------------------------
Метаданные чанка: {'source': 'Resume.csv', 'resume_id': 3547447, 'category': 'BANKING', 'chunk_id': 0}


Распределение длин чанков показывает, что большинство фрагментов имеют размер, близкий к заданному пределу, а значит splitter действительно формирует содержательные блоки текста. При этом наличие очень коротких чанков, как в примере,  указывает на то, что часть служебной информации выделяется отдельно. Мы посчитали, что сейчас это не критично, но в дальнейшем такие чанки можно было бы отфильтровать для повышения точности retrieval.


## 3. Эмбеддинги и векторный индекс FAISS




In [31]:
EMBEDDING_MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
embeddings = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL,encode_kwargs={"normalize_embeddings": True})
vectorstore = FAISS.from_documents(chunked_documents, embeddings)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [32]:
def preview_retrieval(query: str, k: int = 5):
    try:
        results = vectorstore.similarity_search_with_relevance_scores(query, k=k)
        rows = []
        for rank, (doc, score) in enumerate(results, start=1):
            rows.append({
                "rank": rank,
                "score": round(float(score), 4),
                "resume_id": doc.metadata.get("resume_id"),
                "category": doc.metadata.get("category"),
                "chunk_id": doc.metadata.get("chunk_id"),
                "preview": textwrap.shorten(doc.page_content.replace("\n", " "), width=220, placeholder="...")})
        return pd.DataFrame(rows)
    except Exception:
        docs = vectorstore.similarity_search(query, k=k)
        rows = []
        for rank, doc in enumerate(docs, start=1):
            rows.append({
                "rank": rank,
                "score": None,
                "resume_id": doc.metadata.get("resume_id"),
                "category": doc.metadata.get("category"),
                "chunk_id": doc.metadata.get("chunk_id"),
                "preview": textwrap.shorten(doc.page_content.replace("\n", " "), width=220, placeholder="...")})
        return pd.DataFrame(rows)

In [33]:
preview_retrieval("Candidate with Python, machine learning and data analysis experience", k=5)

,rank,score,resume_id,category,chunk_id,preview
0,1,0.4594,12011623,ENGINEERING,1325,. Qualifications Ability to identify uncovered...
1,2,0.3505,12351749,SALES,1545,SALES COORDINATOR Summary Current MS of Data A...
2,3,0.3464,12011623,ENGINEERING,1324,". Experience in SQL, data warehousing, maintai..."
3,4,0.3349,11813872,AGRICULTURE,1155,. Basic Experience in Data Science related tec...
4,5,0.3326,12011623,ENGINEERING,1323,ENGINEERING AND QUALITY TECHNICIAN Career Over...


Пробный поиск показывает, что индекс способен находить фрагменты, тематически связанные с запросом о Python, машинном обучении и анализе данных (как ENGINEERING). Вместе с тем среди топ результатов встречаются документы из категорий, не полностью совпадающих с ожидаемым профилем, что указывает на не идеальное качество retrieval. Следовательно, индекс уже работает, однако результаты всё ещё чувствительны к формулировке запроса и особенностям текста резюме.


## 4. Подключение LLM через Groq



In [38]:
os.environ["GROQ_API_KEY"] = "здесь был наш ключ"
llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0, max_retries=2,)
print("LLM инициализирована.")

LLM инициализирована.



## 5. LangChain RAG-цепочка

Ниже реализован полноценный RAG-пайплайн:

1. вопрос пользователя;
2. семантический поиск top-k чанков;
3. формирование контекста;
4. генерация ответа LLM;
5. обработка случая, когда релевантного контекста нет.

In [42]:
from langchain_core.output_parsers import StrOutputParser
RAG_PROMPT = ChatPromptTemplate.from_template(    """
Ты — помощник по анализу резюме из датасета Resume.csv.
Отвечай только на основе переданного контекста.
Если данных в контексте недостаточно, напиши:
"В найденных резюме недостаточно данных для уверенного ответа."
Правила ответа:
- отвечай на русском языке;
- опирайся только на контекст;
- если уместно, указывай resume_id и category;
- не выдумывай навыки, компании, степени и годы опыта, которых нет в контексте;
- если найдено несколько кандидатов, кратко сравни их.
Контекст:
{context}
Вопрос:
{question}
""")
output_parser = StrOutputParser()
def retrieve_with_scores(query: str, k: int = 5):
    try:
        results = vectorstore.similarity_search_with_relevance_scores(query, k=k)
        return [{"doc": doc, "score": float(score)} for doc, score in results]
    except Exception:
        docs = vectorstore.similarity_search(query, k=k)
        return [{"doc": doc, "score": None} for doc in docs]
MIN_RELEVANCE = 0.30
def select_relevant_chunks(query: str, k: int = 5):
    retrieved = retrieve_with_scores(query, k=k)
    if not retrieved:
        return []
    if retrieved[0]["score"] is None:
        return retrieved
    filtered = [item for item in retrieved if item["score"] >= MIN_RELEVANCE]
    return filtered
def format_context(retrieved_items):
    if not retrieved_items:
        return "Контекст не найден."
    blocks = []
    for rank, item in enumerate(retrieved_items, start=1):
        doc = item["doc"]
        score = item["score"]
        meta = doc.metadata
        header = (
            f"[Чанк {rank}] "
            f"score={round(score, 4) if score is not None else 'n/a'} | "
            f"resume_id={meta.get('resume_id')} | "
            f"category={meta.get('category')} | "
            f"chunk_id={meta.get('chunk_id')}")
        blocks.append(header + "\n" + doc.page_content)
    return "\n\n" + ("\n" + "=" * 100 + "\n\n").join(blocks)
def ask_rag(question: str, k: int = 5):
    retrieved_items = select_relevant_chunks(question, k=k)
    if not retrieved_items:
        return {
            "question": question,
            "answer": "В найденных резюме недостаточно данных для уверенного ответа.",
            "retrieved": []}
    context = format_context(retrieved_items)
    chain = RAG_PROMPT | llm | output_parser
    answer = chain.invoke({"question": question, "context": context})
    return {"question": question, "answer": answer,"retrieved": retrieved_items}

In [44]:
from IPython.display import display, Markdown
sample_result = ask_rag("Find candidates with Python, machine learning and data analysis experience. Summarize the most relevant one.", k=5)
display(Markdown("### Ответ системы"))
display(Markdown(sample_result["answer"]))
print("\nИспользованные чанки:", len(sample_result["retrieved"]))

### Ответ системы

В найденном резюме (resume_id=12011623, category=ENGINEERING) есть опыт работы с Python, машинным обучением и анализом данных. Кандидат имеет навыки в области:

- Python (включая Pandas и Beautiful Soup)
- Машинного обучения (Random Forests, Decision Trees, Boosted Trees, Ridge, LASSO и Elastic Net regression models)
- Анализа данных (Linear Regression, Clustering techniques, logistic regression, Outlier analysis)
- Визуализации данных (Tableau, ggplot)

Этот кандидат имеет опыт работы в качестве инженера и техника по качеству, где он занимался разработкой процессов производства полупроводников, управлением проектами и анализом данных с помощью SQL и JMP. 

В найденных резюме этот кандидат является наиболее подходящим для позиции, требующей опыта работы с Python, машинным обучением и анализом данных.


Использованные чанки: 1


Пробный ответ показывает, что система извлекает признаки, связанные с аналитикой данных и техническими навыками, однако итоговая интерпретация зависит не только от первого найденного чанка, но и от совокупности нескольких возвращенных фрагментов. Мы думаем, что генерация опирается на контекст, но может смещать акцент в сторону наиболее информативного для модели резюме, а не строго наиболее релевантного по рангу.


## 6. Тестирование системы


1. **Фактический** — поиск кандидата с конкретным стеком навыков;
2. **Обобщающий** — типичные компетенции внутри профессионального профиля;
3. **Уточняющий / сравнительный** — пересечение нескольких технологий;
4. **Проверка на отсутствие информации** — запрос, на который контекст, скорее всего, не отвечает.


In [45]:
test_questions = [{"type": "Фактический",
        "question": "Find candidates with Python, machine learning and data analysis experience. What key skills and background are mentioned in the most relevant resumes?"},
    {"type": "Обобщающий",
        "question": "What competencies are typical for HR resumes in this dataset? Summarize the recurring skills based on retrieved examples."},
    {"type": "Уточняющий / сравнительный",
        "question": "Are there candidates whose resumes mention Java, SQL and Spring together? Compare the most relevant ones."},
    {"type": "Проверка отсутствия информации",
        "question": "What salary expectations do the candidates mention in their resumes?"},]

In [46]:
results = []
for item in test_questions:
    result = ask_rag(item["question"], k=5)
    result["type"] = item["type"]
    results.append(result)
print("Готово. Получено результатов:", len(results))

Готово. Получено результатов: 4


In [47]:
for idx, result in enumerate(results, start=1):
    display(Markdown(f"## Вопрос {idx}: {result['type']}"))
    display(Markdown(f"**Вопрос:** {result['question']}"))
    display(Markdown("**Ответ системы:**"))
    display(Markdown(result["answer"]))
    if result["retrieved"]:
        rows = []
        for rank, item in enumerate(result["retrieved"], start=1):
            doc = item["doc"]
            rows.append({
                "rank": rank,
                "score": round(item["score"], 4) if item["score"] is not None else None,
                "resume_id": doc.metadata.get("resume_id"),
                "category": doc.metadata.get("category"),
                "chunk_id": doc.metadata.get("chunk_id"),
                "preview": textwrap.shorten(doc.page_content.replace("\n", " "), width=240, placeholder="...")})
        display(pd.DataFrame(rows))
    else:
        display(Markdown("_Релевантные чанки не прошли порог отбора._"))
    print("-" * 120)

## Вопрос 1: Фактический

**Вопрос:** Find candidates with Python, machine learning and data analysis experience. What key skills and background are mentioned in the most relevant resumes?

**Ответ системы:**

В найденных резюме есть несколько кандидатов с опытом работы с Python, машинным обучением и анализом данных. 

Одним из наиболее подходящих кандидатов является тот, у кого resume_id=12415691 и category=DESIGNER. В его резюме упоминается опыт работы с Python, а также с другими языками программирования, такими как C, C++, C#.NET, Java, PHP, Mathematica, Oracle, PL/SQL, MySQL. Кроме того, он имеет опыт работы с базами данных и системами управления, такими как Oracle Database Administration (11g), MySQL database administration, Ellucian Banner ERP, и Oracle Application Express(APEX).

Другим кандидатом является тот, у кого resume_id=12011623 и category=ENGINEERING. В его резюме упоминается опыт работы с SAS, Web scraping, SQL, Predictive modelling и data visualization. Он также имеет опыт работы с данными, включая их очистку, обработку и моделирование.

Оба кандидата имеют сильный технический фон и опыт работы с данными, что делает их подходящими для позиций, требующих опыта работы с Python, машинным обучением и анализом данных. Однако, первый кандидат имеет более широкий спектр технических навыков, включая опыт работы с различными языками программирования и базами данных.

,rank,score,resume_id,category,chunk_id,preview
0,1,0.4458,12334140,INFORMATION-TECHNOLOGY,1508,". This way, the resume reviewer can see the ca..."
1,2,0.4419,10734870,CONSTRUCTION,509,CONSTRUCTION Summary The purpose of submitting...
2,3,0.4234,12415691,DESIGNER,1587,INFORMATION DESIGNER Summary of Qualifications...
3,4,0.4109,10541358,BUSINESS-DEVELOPMENT,339,. Recruitment of top talent for both entry lev...
4,5,0.3984,12011623,ENGINEERING,1323,ENGINEERING AND QUALITY TECHNICIAN Career Over...


...

**Вопрос:** What salary expectations do the candidates mention in their resumes?

**Ответ системы:**

В найденных резюме недостаточно данных для уверенного ответа.

,rank,score,resume_id,category,chunk_id,preview
0,1,0.3435,11270462,DIGITAL-MEDIA,840,. The proposal was to increase recruitment num...
1,2,0.3332,10541358,BUSINESS-DEVELOPMENT,339,. Recruitment of top talent for both entry lev...
2,3,0.3143,12334140,INFORMATION-TECHNOLOGY,1507,. Mention specifically how your skills and exp...
3,4,0.3143,23933031,BPO,2216,"; +/-2,000 workforce Analyzed, monitored and r..."
4,5,0.3061,12334140,INFORMATION-TECHNOLOGY,1508,". This way, the resume reviewer can see the ca..."


------------------------------------------------------------------------------------------------------------------------


В фактическом запросе система обнаружила несколько резюме с упоминанием аналитических и технических навыков. При этом среди топ результатов присутствуют и частично нерелевантные категории, что показывает ограничение семантического поиска на коротких и неоднородных фрагментах. Тем не менее ответ модели в целом отражает данные из контекста и выделяет кандидатов с техническим профилем.

В обобщающем вопросе retrieval оказался более устойчивым, поскольку запрос был ориентирован на повторяющиеся HR-компетенции. Большинство релевантных чанков действительно относится к категории HR, поэтому модель смогла выделить типичные функции, связанные с подбором персонала, консультированием и HR-администрированием. Этот пример показывает, что RAG особенно хорошо работает на задачах агрегирования часто повторяющихся признаков.

В сравнительном вопросе система сработала как бы осторожно и сообщила о недостаточности данных для уверенного ответа. Наверное, такое поведение можно считать корректным, поскольку retrieval нашёл только частичные совпадения по отдельным технологиям, но не дал достаточно надёжного контекста для уверенного утверждения о совместном наличии Java, SQL и Spring. Данный пример показывает, что механизм фильтрации помогает сдерживать необоснованные выводы модели.

Запрос о salary expectations демонстрирует проверку на отсутствие информации. Хотя семантический поиск вернул несколько фрагментов с частично похожей лексикой, в retrieved-контексте не оказалось прямых сведений о зарплатных ожиданиях. Следовательно, отказ от ответа здесь является правильным и показывает, что система не делает догадки.


## 7. Качественный анализ результатов

Ниже строится краткий автоматический анализ: какие чанки использовались, почему они могли быть выбраны, и насколько ответ выглядит надёжным.


In [48]:
STOPWORDS = {"the", "and", "for", "with", "from", "that", "this", "have", "their", "they","what", "are", "there", "whose", "mention", "mentioned", "into", "your","как", "что", "это", "для", "или", "есть", "ли", "по", "на", "в", "и"}
def keyword_overlap(question: str, text: str, top_n: int = 8):
    q_tokens = re.findall(r"[A-Za-zА-Яа-я\-\+]{3,}", question.lower())
    q_tokens = [t for t in q_tokens if t not in STOPWORDS]
    text_lower = text.lower()
    hits = []
    for token in q_tokens:
        if token in text_lower and token not in hits:
            hits.append(token)
    return hits[:top_n]
def build_analysis(result):
    if not result["retrieved"]:
        return (f"**Анализ:** для вопроса типа **{result['type']}** после retrieval не было найдено "
            f"достаточно релевантных чанков выше порога `{MIN_RELEVANCE}`. "
            f"Это корректное поведение для сценария, где в контексте действительно может отсутствовать ответ.")
    top_ids = []
    top_categories = []
    justifications = []
    for item in result["retrieved"][:3]:
        doc = item["doc"]
        meta = doc.metadata
        top_ids.append(str(meta.get("resume_id")))
        top_categories.append(str(meta.get("category")))
        overlap = keyword_overlap(result["question"], doc.page_content)
        if overlap:
            justifications.append(f"resume_id={meta.get('resume_id')} ({meta.get('category')}): совпали признаки {', '.join(overlap)}")
        else:
            justifications.append(f"resume_id={meta.get('resume_id')} ({meta.get('category')}): чанк выбран по семантической близости без явного лексического пересечения")
    analysis = f"""
**Анализ:**
- Тип вопроса: **{result['type']}**
- Использованы чанки резюме: **{", ".join(top_ids)}**
- Категории найденных кандидатов: **{", ".join(top_categories)}**
- Почему выбраны именно эти чанки:
  - """ + "\n  - ".join(justifications) + f"""
**Промежуточный вывод:** если ответ ссылается на навыки и опыт, которые действительно видны в top-k чанках, значит RAG-пайплайн работает корректно: сначала retrieval находит релевантный контекст, затем LLM формирует итоговый ответ на его основе.
"""
    return analysis
for idx, result in enumerate(results, start=1):
    display(Markdown(f"### Анализ для вопроса {idx}"))
    display(Markdown(build_analysis(result)))

### Анализ для вопроса 1


**Анализ:**  
- Тип вопроса: **Фактический**  
- Использованы чанки резюме: **12334140, 10734870, 12415691**  
- Категории найденных кандидатов: **INFORMATION-TECHNOLOGY, CONSTRUCTION, DESIGNER**  
- Почему выбраны именно эти чанки:  
  - resume_id=12334140 (INFORMATION-TECHNOLOGY): совпали признаки experience, skills, relevant
  - resume_id=10734870 (CONSTRUCTION): совпали признаки experience, skills
  - resume_id=12415691 (DESIGNER): совпали признаки python, data, experience, skills, background
**Промежуточный вывод:** если ответ ссылается на навыки и опыт, которые действительно видны в top-k чанках, значит RAG-пайплайн работает корректно: сначала retrieval находит релевантный контекст, затем LLM формирует итоговый ответ на его основе.


### Анализ для вопроса 2


**Анализ:**  
- Тип вопроса: **Обобщающий**  
- Использованы чанки резюме: **11847784, 12334140, 13879043**  
- Категории найденных кандидатов: **HR, INFORMATION-TECHNOLOGY, HR**  
- Почему выбраны именно эти чанки:  
  - resume_id=11847784 (HR): совпали признаки skills, based
  - resume_id=12334140 (INFORMATION-TECHNOLOGY): совпали признаки typical, skills
  - resume_id=13879043 (HR): чанк выбран по семантической близости без явного лексического пересечения
**Промежуточный вывод:** если ответ ссылается на навыки и опыт, которые действительно видны в top-k чанках, значит RAG-пайплайн работает корректно: сначала retrieval находит релевантный контекст, затем LLM формирует итоговый ответ на его основе.


### Анализ для вопроса 3


**Анализ:**  
- Тип вопроса: **Уточняющий / сравнительный**  
- Использованы чанки резюме: **12334140, 11890896, 13964744**  
- Категории найденных кандидатов: **INFORMATION-TECHNOLOGY, ENGINEERING, BPO**  
- Почему выбраны именно эти чанки:  
  - resume_id=12334140 (INFORMATION-TECHNOLOGY): совпали признаки relevant
  - resume_id=11890896 (ENGINEERING): совпали признаки sql
  - resume_id=13964744 (BPO): совпали признаки java, sql
**Промежуточный вывод:** если ответ ссылается на навыки и опыт, которые действительно видны в top-k чанках, значит RAG-пайплайн работает корректно: сначала retrieval находит релевантный контекст, затем LLM формирует итоговый ответ на его основе.


### Анализ для вопроса 4


**Анализ:**  
- Тип вопроса: **Проверка отсутствия информации**  
- Использованы чанки резюме: **11270462, 10541358, 12334140**  
- Категории найденных кандидатов: **DIGITAL-MEDIA, BUSINESS-DEVELOPMENT, INFORMATION-TECHNOLOGY**  
- Почему выбраны именно эти чанки:  
  - resume_id=11270462 (DIGITAL-MEDIA): чанк выбран по семантической близости без явного лексического пересечения
  - resume_id=10541358 (BUSINESS-DEVELOPMENT): чанк выбран по семантической близости без явного лексического пересечения
  - resume_id=12334140 (INFORMATION-TECHNOLOGY): чанк выбран по семантической близости без явного лексического пересечения
**Промежуточный вывод:** если ответ ссылается на навыки и опыт, которые действительно видны в top-k чанках, значит RAG-пайплайн работает корректно: сначала retrieval находит релевантный контекст, затем LLM формирует итоговый ответ на его основе.


Промежуточный анализ показывает, что retrieval обычно извлекает чанки, содержащие часть ключевых признаков из вопроса. Это подтверждает работоспособность, хотя степень точности зависит от того, насколько специфичны формулировки запроса и насколько явно нужные навыки выражены в тексте резюме.


## 8. Итоговый вывод по качеству RAG-системы


# Финальный аналитический вывод
В этом проекте была построена RAG-система по датасету Resume.csv.
Система загружает и очищает резюме, разбивает их на чанки, индексирует их в FAISS и использует Groq-модель для генерации ответа на основе найденного контекста.

Что получилось хорошо:
загружено и подготовлено более 100 записей;

чанкинг подходит для длинных полу-структурированных резюме;

retrieval возвращает не только текст, но и полезные метаданные (resume_id, category, chunk_id);

ответы опираются на top-k чанки;

предусмотрен сценарий отсутствия достаточного контекста.


Ограничения:

резюме написаны в свободной форме, поэтому навыки могут называться по-разному;

один и тот же кандидат может содержать много технологий в разных частях текста, поэтому качество ответа зависит от того, какие именно чанки попали в top-k;

В ходе тестирования система во всех случаях смогла извлечь набор семантически близких чанков, однако не для каждого вопроса этого контекста оказалось достаточно для уверенного ответа. Это показывает, что один факт наличия retrieved-фрагментов ещё не гарантирует полноты и точности итоговой генерации.


В целом разработанная система демонстрирует базовую работоспособность RAG-подхода на корпусе резюме. Наиболее надёжные результаты получаются для запросов о типичных навыках и профессиональных компетенциях, тогда как сложные сравнительные вопросы требуют более точного retrieval.